In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")
device = torch.device("cuda:0")

# GIFTBench
GIFT_BASE_MODEL    = "runwayml/stable-diffusion-v1-5"
GIFT_LORA_IMAGENET = "/home/weissl/PycharmProjects/genai_tigs/sd_weights/teddy_bear_imagenet-000005.safetensors"
GIFT_LORA_CELEBA   = "/home/weissl/PycharmProjects/genai_tigs/sd_weights/celeb-lora-512-all-000002.safetensors"
GIFT_STEPS = 25

# GIFTBench stats paths
GIFT_IMAGENET_STATS = "/home/weissl/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_/perturbresult/*.json"
GIFT_CELEBA_STATS   = "/home/weissl/PycharmProjects/genai_tigs/sd_weights/celebahq_generatorlow/*.json"
GIFT_DRIVING_STATS  = "/home/weissl/PycharmProjects/genai_tigs/yolo_sd/test_/perturbresult/*.json"

In [2]:
from _flop import (
    count_flops, count_flops_stylegan, count_flops_and_trainable,
    SynthWrapper, SiTOneStepWrapper, UNet2DOneStepWrapper, SDCNOneStepWrapper,
    VaeDecodeWrapper, fix_control_projector, sd_inference_flops, load_budgets,
    save_flops, load_flops,
)

## Mimicry — StyleGAN Synthesis FLOPs
Might need different Conda Env to Run!

In [3]:
import os
os.environ["CUDA_HOME"]    = "/usr/local/cuda"
os.environ["PATH"]         = f"/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["CPATH"]        = f"/usr/local/cuda/include:" + os.environ.get("CPATH", "")
os.environ["LIBRARY_PATH"] = f"/usr/local/cuda/lib64:" + os.environ.get("LIBRARY_PATH", "")

from src.manipulator.style_gan_manipulator import StyleGANManipulator

def stylegan_flops(pkl, mix_size, batch_size=1):
    m = StyleGANManipulator(pkl, device, mix_size, interpolate=True, batch_size=batch_size)
    w = m.get_w(0, 0)
    f = count_flops_stylegan(SynthWrapper(m._generator.synthesis), w)
    del m; torch.cuda.empty_cache()
    return f

flops_synth_imagenet = stylegan_flops("../models/generators/imagenet256.pkl", (1, 15))
flops_synth_celeba   = stylegan_flops("../models/generators/stylegan2-ffhq-1024x1024.pkl", (0, 8), batch_size=1)
save_flops("mimicry_imagenet", flops_synth_imagenet)
save_flops("mimicry_celeba",   flops_synth_celeba)
print(f"Mimicry ImageNet: {flops_synth_imagenet/1e12:.3f} TFLOPs")
print(f"Mimicry CelebA:   {flops_synth_celeba/1e12:.3f} TFLOPs")

Setting up PyTorch plugin "bias_act_plugin"... Done.
Setting up PyTorch plugin "filtered_lrelu_plugin"... Done.
Setting up PyTorch plugin "upfirdn2d_plugin"... Done.
Mimicry ImageNet: 1.497 TFLOPs  → _mimicry_imagenet.json
Mimicry CelebA:   0.074 TFLOPs  → _mimicry_celeba.json


## HyNeA — Hypernet + VAE FLOPs

One inference = one `manipulate()` call + one `get_images()` (VAE decode).

**Effective FLOPs formula**: `2 × f_forward + f_trainable_only`
- Frozen backbone (SiT/UNet) and VAE: only activation gradients needed → 1× extra backward per frozen part
- Trainable hypernet (control layers, zero layers): full backward → 2× extra
- Combined: `forward + (1× frozen_backward + 2× trainable_backward)` = `2×f_forward + f_trainable_only`

In [3]:
from src.manipulator.diffusion_manipulator import SitHyNeAManipulator, LDMHyNeAManipulator, SDCNHyNeAManipulator

In [4]:
m   = SitHyNeAManipulator("../models/generators/ldm_im.pt", device=device, control_shape=(1000,))
b   = 2 if m._cfg > 1.0 else 1
xt  = torch.randn(b, m._in_channels, m._latent_size, m._latent_size, device=device)
inp = (xt, torch.full((b,), 0.5, device=device), m._embed_y([0]*b), torch.zeros(b, 1000, device=device))

f_step, f_train = count_flops_and_trainable(SiTOneStepWrapper(m._hyper_net), inp)
f_vae           = count_flops(VaeDecodeWrapper(m._vae), xt[:1])
del m; torch.cuda.empty_cache()

flops_hynea_imagenet           = f_step * 50 + f_vae   # 50 = default n_steps
flops_hynea_imagenet_effective = 2 * flops_hynea_imagenet + f_train * 50
save_flops("hynea_imagenet", flops_hynea_imagenet, flops_hynea_imagenet_effective)
print(f"HyNeA ImageNet — fwd: {flops_hynea_imagenet/1e12:.3f} TFLOPs | eff: {flops_hynea_imagenet_effective/1e12:.3f} TFLOPs")

Using cache found in /home/weissl/.cache/torch/hub/facebookresearch_dinov2_main


HyNeA ImageNet — fwd: 15.183 TFLOPs | eff: 33.906 TFLOPs  → _hynea_imagenet.json


In [5]:
m    = LDMHyNeAManipulator(control_shape=(40,), device=device, diffusion_steps=100)
in_ch, ss = m._model.config["in_channels"], m._model.sample_size
xt   = torch.randn(1, in_ch, ss, ss, device=device)
ctrl = torch.zeros(1, 40, device=device)
t    = torch.tensor([500], dtype=torch.long, device=device)
step_w = UNet2DOneStepWrapper(m._hyper_net)
fix_control_projector(step_w)

f_step, f_train = count_flops_and_trainable(step_w, (xt, ctrl, t))
f_vae           = count_flops(VaeDecodeWrapper(m._vae), xt)
del m, step_w; torch.cuda.empty_cache()

flops_hynea_celeba           = f_step * 100 + f_vae
flops_hynea_celeba_effective = 2 * flops_hynea_celeba + f_train * 100
save_flops("hynea_celeba", flops_hynea_celeba, flops_hynea_celeba_effective)
print(f"HyNeA CelebA   — fwd: {flops_hynea_celeba/1e12:.3f} TFLOPs | eff: {flops_hynea_celeba_effective/1e12:.3f} TFLOPs")

The config attributes {'timestep_values': None, 'timesteps': 1000} were passed to DDIMScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.


HyNeA CelebA   — fwd: 12.681 TFLOPs | eff: 28.110 TFLOPs  → _hynea_celeba.json


In [4]:
CONTROLNET_PATH = "/home/weissl/Projects/SMOO/models/generators/fortuna"
N_STEPS = 50

yolo_det = int(sum((512/s)**2 for s in [32,16,8]))
m    = SDCNHyNeAManipulator("runwayml/stable-diffusion-v1-5", (80, yolo_det), controlnet_path=CONTROLNET_PATH, device=device)
xt          = torch.randn(1, 4, 64, 64, device=device)
ctrl        = torch.zeros(1, 80, yolo_det, device=device)
y_embed     = torch.zeros(2, 77, 768, device=device)        # 2× for CFG; SD v1-5 CLIP dim
t           = torch.tensor([500], dtype=torch.long, device=device)
ctrl_signal = torch.randn(2, 3, 512, 512, device=device)   # seg map, 2× for CFG

# SDCNOneStepWrapper already disables xformers on controlnet + unet;
# the VAE also uses xformers and must be patched separately.
m._pipe.vae.disable_xformers_memory_efficient_attention()

step_w = SDCNOneStepWrapper(m._hyper_net)
f_step, f_train = count_flops_and_trainable(step_w, (xt, y_embed, t, ctrl, ctrl_signal))
f_vae           = count_flops(VaeDecodeWrapper(m._pipe.vae), xt)
del m, step_w; torch.cuda.empty_cache()

flops_hynea_driving           = f_step * N_STEPS + f_vae
flops_hynea_driving_effective = 2 * flops_hynea_driving + f_train * N_STEPS
save_flops("hynea_driving", flops_hynea_driving, flops_hynea_driving_effective)
save_flops("gift_driving",  flops_hynea_driving)   # forward-only; same generator as HyNeA driving
print(f"HyNeA Driving  — fwd: {flops_hynea_driving/1e12:.3f} TFLOPs | eff: {flops_hynea_driving_effective/1e12:.3f} TFLOPs")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

HyNeA Driving  — fwd: 58.100 TFLOPs | eff: 127.496 TFLOPs


## GIFTBench — Full SD Denoising FLOPs

One inference = `n_steps × UNet forward` + `VAE decode` (no backprop — inference only).

Fill in `GIFTBENCH_*_CKPT` and `GIFTBENCH_STEPS` in Cell 1 before running.

In [5]:
flops_gift_sd, *_ = sd_inference_flops(GIFT_BASE_MODEL, GIFT_STEPS, device, lora_path=GIFT_LORA_IMAGENET)
save_flops("gift_imagenet", flops_gift_sd)
save_flops("gift_celeba",   flops_gift_sd)
print(f"GIFTBench SD v1-5: {flops_gift_sd/1e12:.3f} TFLOPs")

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


GIFTBench SD v1-5: 10.559 TFLOPs


In [6]:
usr_path = "/home/weissl"

GIFT_EFF_STATS = f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_eff/perturbresult/*.json"
GIFT_VIT_STATS = f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_vit/perturbresult/*.json"

MIM_EXTRA = ("w0_trials", "wn_trials")

# (label, budget_glob, budget_field, extra_fields, flop_key)
# eff/vit reuse base flop_key — same generator architecture, only SUT backbone differs
CONFIGS = [
    ("mimicry_imagenet",     f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_class*/"     "stats.json", "budget_used", MIM_EXTRA, "mimicry_imagenet"),
    ("mimicry_imagenet_eff", f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_eff_class*/" "stats.json", "budget_used", MIM_EXTRA, "mimicry_imagenet"),
    ("mimicry_imagenet_vit", f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_vit_class*/" "stats.json", "budget_used", MIM_EXTRA, "mimicry_imagenet"),
    ("mimicry_celeba",       f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*/"         "stats.json", "budget_used", MIM_EXTRA, "mimicry_celeba"),
    ("hynea_imagenet",       f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*/"              "stats.json", "budget_used", (),        "hynea_imagenet"),
    ("hynea_imagenet_eff",   f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_eff/*/"         "stats.json", "budget_used", (),        "hynea_imagenet"),
    ("hynea_imagenet_vit",   f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_vit/*/"         "stats.json", "budget_used", (),        "hynea_imagenet"),
    ("hynea_celeba",         f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*/"              "stats.json", "budget_used", (),        "hynea_celeba"),
    ("hynea_driving",        f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*/"              "stats.json", "budget_used", (),        "hynea_driving"),
    ("gift_imagenet",        GIFT_IMAGENET_STATS, "budget", (), "gift_imagenet"),
    ("gift_imagenet_eff",    GIFT_EFF_STATS,       "budget", (), "gift_imagenet"),
    ("gift_imagenet_vit",    GIFT_VIT_STATS,       "budget", (), "gift_imagenet"),
    ("gift_celeba",          GIFT_CELEBA_STATS,    "budget", (), "gift_celeba"),
    ("gift_driving",         GIFT_DRIVING_STATS,   "budget", (), "gift_driving"),
]

print(f"{'Config':<30} {'Mean TFLOPs':>14} {'Std':>10}")
print("-" * 56)
for name, pattern, field, extra, flop_key in CONFIGS:
    b = load_budgets(pattern, field=field, extra_fields=extra)
    _, eff = load_flops(flop_key)
    if not len(b) or eff is None:
        print(f"{name:<30}  no data")
        continue
    tf = b * eff / 1e12
    print(f"{name:<30} {tf.mean():>14.1f} {tf.std():>10.1f}")

Config                            Mean TFLOPs        Std
--------------------------------------------------------
mimicry_imagenet                       3739.0      581.6
mimicry_imagenet_eff                   3719.6      551.6
mimicry_imagenet_vit                   3785.3      434.2
mimicry_celeba                          198.5        2.8
hynea_imagenet                          857.5      901.3
hynea_imagenet_eff                      863.9      954.7
hynea_imagenet_vit                      561.1      575.0
hynea_celeba                            851.7      879.0
hynea_driving                           728.5      718.0
gift_imagenet                          7387.1     7693.2
gift_imagenet_eff                      2977.5     2214.2
gift_imagenet_vit                      3360.3     2756.2
gift_celeba                           18963.3    11295.8
gift_driving                          13486.4    14130.0
